在 PyTorch 中，<font color='red'>默认情况下所有的 GPU 操作都会被提交到“默认流（Default Stream）”中按顺序排队执行。</font>这就好比在超市只开了一个收银台，所有人必须排一队结账。

- 为了榨干 GPU 的极致性能，PyTorch 提供了 torch.cuda.Stream 和 torch.cuda.Event，让我们能够开设“多个收银台”并精确记录每个顾客的处理时间。

## 🚀 torch.cuda.Stream：开设多个并行通道
###### 核心解释：
Stream 是 GPU 上的一个命令队列。
- <font color='red'>通过创建多个流，我们可以让互不依赖的任务（比如数据拷贝和模型计算）在不同的流上并发执行。</font>这能极大地隐藏数据传输的延迟，提升整体吞吐量。
###### 实战场景： 假设我们要处理两个完全独立的大矩阵乘法，用单流只能串行计算，用双流则可以让 GPU 同时开工。

### ⚙️ 2. 为什么能“同时”执行？（Hyper-Q 技术）
现代 NVIDIA GPU（如 Ampere 架构的 A100 等）内部配备了多个硬件工作队列（例如 128 个 Hyper-Q 硬件队列）。当你创建多个 torch.cuda.Stream 时，PyTorch 会将这些流映射到不同的硬件队列上，从而让 GPU 能够真正同时处理多个独立的任务。

虽然理论上可以并行，但在实际工程中，你需要了解以下两个限制：
- <font color='red'>启动开销导致的时间差</font>：CPU 在向 GPU 的不同流下发任务时，会有一定的调度开销。因此，即使你写了代码让两个流同时跑，它们在 GPU 上的实际启动时间往往会有微小的偏差，呈现出“错开启动”的状态，而不是精准的毫秒级同步。
- 硬件资源瓶颈：如果单个任务已经把 GPU 的计算单元或显存带宽占满了（例如一个巨大的矩阵乘法吃满了所有 SM），那么即使你开了 10 个流，其他流也只能排队等待资源释放，无法实现真正的物理并行。

In [ ]:
import torch

if torch.cuda.is_available():
    # 创建两个独立的 CUDA 流
    stream1 = torch.cuda.Stream()
    stream2 = torch.cuda.Stream()

    # 准备两组独立的数据
    data1 = torch.randn(5000, 5000, device='cuda')
    data2 = torch.randn(5000, 5000, device='cuda')

    # --- 在流1中执行第一个任务 ---
    with torch.cuda.stream(stream1):
        result1 = torch.matmul(data1, data1)
    
    # --- 在流2中执行第二个任务 ---
    with torch.cuda.stream(stream2):
        result2 = torch.matmul(data2, data2)

    # 关键：由于两个流是并行的，CPU下发完指令后会立刻跑到这里。
    # 如果现在直接打印结果，GPU可能还没算完。
    # 必须等待两个流都执行完毕，才能安全地获取结果。
    stream1.synchronize()
    stream2.synchronize()
    
    print("两个独立的计算任务已全部在GPU上并行完成！")

## 🎯 torch.cuda.Event：流中的精准打点器
###### 核心解释：
由于 GPU 是异步执行的，用 CPU 的 time.time() 根本无法准确测量 GPU 代码块的真实耗时。
- <font color='red'>Event 就像一个精准的“打点器”，可以在某个流中插入一个时间戳标记，配合 elapsed_time() 就能算出两点之间 GPU 实际工作的毫秒数。</font>
###### 实战场景： 精确测量上面流1中矩阵乘法的真实耗时。

In [ ]:
import torch

if torch.cuda.is_available():
    # 创建启用计时功能的 Event
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    stream = torch.cuda.Stream()
    data = torch.randn(8000, 8000, device='cuda')

    # 将计算任务放入自定义流中
    with torch.cuda.stream(stream):
        start_event.record()  # 在流中打下“开始”标记
        result = torch.matmul(data, data)
        end_event.record()    # 在流中打下“结束”标记

    # 等待该流中的所有任务（包括计算和打点）执行完毕
    stream.synchronize()
    
    # 计算两个标记点之间的 GPU 实际耗时（单位：毫秒）
    elapsed_time_ms = start_event.elapsed_time(end_event)
    print(f"流中矩阵乘法在 GPU 上的真实耗时: {elapsed_time_ms:.2f} ms")

## 🤝 进阶联动：用 Event 协调多个 Stream
- <font color='red'>在实际的深度学习训练中，最经典的用法是用 Event 来协调“数据传输流”和“计算流”。</font>
- 比如，当 GPU 正在计算流里拼命训练第 N 批数据时，我们可以让传输流提前把第 N+1 批数据从 CPU 搬运到显存。为了让计算流知道数据什么时候搬运完毕，我们可以这样做：

In [ ]:
# 伪代码逻辑演示
compute_stream = torch.cuda.Stream()
transfer_stream = torch.cuda.Stream()

data_ready_event = torch.cuda.Event() # 用于传递信号的 Event

# 1. 在传输流中异步搬运下一批数据
with torch.cuda.stream(transfer_stream):
    next_batch_gpu = next_batch_cpu.to('cuda', non_blocking=True)
    data_ready_event.record() # 搬运完立刻打个标记：数据准备好了！

# 2. 在计算流中，先等待数据准备就绪的信号，再开始计算
with torch.cuda.stream(compute_stream):
    compute_stream.wait_event(data_ready_event) # 没收到信号前，计算流会在这里乖乖等待
    output = model(next_batch_gpu) # 收到信号，数据已就绪，立刻开始计算！

- 通过这种“流 + 事件”的精细配合，就能实现数据传输与模型计算的完美重叠，最大化 GPU 的利用率。

## 💡 为什么需要 Event？
- 如果不使用 Event 进行协调，由于 stream_A 和 stream_B 是异步并行执行的，stream_B 可能会在 stream_A 还没算出 result_A 的时候就开始执行乘法操作，从而导致读取到错误的数据或引发程序崩溃。
- 通过 event_A_done.record() 和 stream_B.wait_event()，我们既保留了两个流各自独立并发的优势，又在存在数据依赖的关键节点上建立了精准的“握手”机制，确保了程序的正确性和执行效率。

In [ ]:
import torch

# 确保 CUDA 可用
if not torch.cuda.is_available():
    print("CUDA 不可用")
else:
    # 1. 创建两个独立的 CUDA 流
    stream_A = torch.cuda.Stream()
    stream_B = torch.cuda.Stream()

    # 2. 创建一个 Event（用于同步的“路标”）
    event_A_done = torch.cuda.Event()

    # 3. Stream A 的操作：负责准备数据
    with torch.cuda.stream(stream_A):
        print("流 A：开始准备数据...")
        # 模拟一个耗时的计算任务
        data_A = torch.randn(2048, 2048, device='cuda')
        result_A = data_A @ data_A
        print("流 A：数据准备完成，即将插上路标（记录 Event）。")
        # 【关键步骤1】在流 A 上记录事件，标记前面的计算已经全部完成
        event_A_done.record(stream_A)

    # 4. Stream B 的操作：必须依赖流 A 的结果
    with torch.cuda.stream(stream_B):
        # 【关键步骤2】流 B 遇到路标，必须等待流 A 记录完 event_A_done 才能继续
        stream_B.wait_event(event_A_done)
        
        print("流 B：检测到流 A 的路标，开始使用流 A 的结果进行后续计算...")
        # 这里安全地使用流 A 算出的 result_A
        final_result = result_A * 2
        print("流 B：依赖计算完成。")

    # 5. 强制同步，确保 CPU 等待所有 GPU 任务执行完毕，以便观察打印顺序
    torch.cuda.synchronize()
    print("--- 所有 GPU 任务全部完成 ---")